# Lasso Experiment

This notebook covers part **G**: fitting Lasso regression on the Runge function by
gradient descent — there is no closed form for the L1-penalized problem, unlike
OLS/Ridge. It reuses the exact same `GradientDescent`/`Optimizer` machinery as parts
E/F (`penalty="l1"`; see `Lasso`, a thin convenience subclass), on the same degree-8
Runge dataset as `gradient_descent_experiment.ipynb` (same $n$, noise, seed).

One difference from that notebook: here Lasso is fit **without a bias term**, the
same way parts A-D fit OLS/Ridge
(`regression.degree_sweep.fit_polynomial_degree_sweep`): no intercept column in the
design matrix, every column standardized, and the target centered
(`y_train - y_train.mean()`) rather than an intercept fit as an extra, unpenalized
coefficient. This keeps the L1 penalty acting on every fitted coefficient, with
nothing carved out as a special case.

Three questions, matching the GitHub issue:

- **Non-differentiability at zero**: `|theta_j|` isn't differentiable at
  `theta_j = 0`. What does JAX's autodiff return there, and is it a valid subgradient?
- **Does Lasso inherit OLS's or Ridge's convergence behavior** under gradient
  descent, and why?
- **Comparison with scikit-learn**, converting carefully between this package's `lam`
  and scikit-learn's `alpha`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Lasso as SkLasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data
from fys_stk4155_p1.regression.autodiff import lasso_autodiff_gradient
from fys_stk4155_p1.regression.cost import (
    hessian_max_eigenvalue,
    lasso_subgradient,
    sklearn_alpha_to_lam,
)
from fys_stk4155_p1.regression.gradient_descent import GradientDescent
from fys_stk4155_p1.regression.lasso import Lasso
from fys_stk4155_p1.regression.shrinkage import lasso_coefficient_path

## Setup: data, design matrix, scaling, centering

No intercept column (`intercept=False`): every column of the design matrix is a
genuine polynomial term, `x^1 ... x^degree`, standardized on the training split. The
target is centered by its training-split mean; since standardized features have zero
mean, that mean is exactly the intercept the model would otherwise have fit — it's
just recovered separately (`y_train.mean()`) instead of as a `Lasso` coefficient.

In [ ]:
def standardize(X_train, X_test):
    """Standardize every column (fit on train only); no intercept column to skip."""
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test)


x, y = generate_runge_data(n=200, noise_std=0.1, seed=42)
X_full = univariate_polynomial_design_matrix(x=x, degree=8, intercept=False)

X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)
X_train, X_test = standardize(X_train, X_test)

y_mean = y_train.mean()
y_train_c = y_train - y_mean

## The L1 subgradient at theta_j = 0

`|theta_j|` isn't differentiable at `theta_j = 0`; its subdifferential there is the
interval $[-1, 1]$. `lasso_subgradient` uses `np.sign(0) == 0`, the zero-valued
member of that interval; JAX's autodiff (`lasso_autodiff_gradient`) instead returns
exactly `+1.0` at 0 — also a member of $[-1, 1]$, so also a valid subgradient, just a
different, arbitrary choice baked into how JAX's `abs` primitive is implemented. The
two should therefore agree everywhere except exactly at a zero coefficient.

In [ ]:
theta = np.array([0.3, -1.2, 0.0, 2.5, -0.7, 0.05, 1.1, -0.2])  # theta[2] = 0.0 exactly
lam = 0.1

analytical = lasso_subgradient(X_train, y_train_c, theta, lam=lam)
autodiff = lasso_autodiff_gradient(X_train, y_train_c, theta, lam=lam)

elsewhere = np.delete(np.abs(analytical - autodiff), 2)
gap = autodiff[2] - analytical[2]
print(f"analytical[2] (np.sign(0) == 0):      {analytical[2]:.6f}")
print(f"autodiff[2]   (JAX's choice at 0):     {autodiff[2]:.6f}")
print(f"autodiff[2] - analytical[2]:           {gap:.6f}  (should be exactly lam = {lam})")
print(f"max |difference| at every other index: {elsewhere.max():.2e}  (should be ~0)")

Exactly as predicted: the two gradients agree to numerical precision everywhere
except at index 2, where they differ by precisely `lam` — the gap between the two
subgradient choices, `1.0 - 0.0`, scaled by the penalty strength. Neither choice is
"more correct"; scikit-learn's coordinate-descent solver sidesteps the question
entirely via closed-form soft-thresholding rather than picking a subgradient.

## Sparsity path

As $\lambda$ grows, Lasso drives coefficients toward zero — its headline property,
and the reason it's used for feature selection where Ridge (which shrinks but never
zeroes) is not. This section uses a well-conditioned degree (4, condition number
~50 — see `gradient_descent_experiment.ipynb`'s degree-sensitivity section) so the
path below reflects the L1 penalty's effect, not an optimizer struggling to converge.

In [ ]:
X_full_p4 = univariate_polynomial_design_matrix(x=x, degree=4, intercept=False)
X_train_p4, X_test_p4, y_train_p4, _ = train_test_split(
    X_full_p4, y, test_size=0.2, random_state=42
)
X_train_p4, X_test_p4 = standardize(X_train_p4, X_test_p4)
y_train_p4_c = y_train_p4 - y_train_p4.mean()

lambdas_path = np.logspace(-3, 1, 25)
path = lasso_coefficient_path(
    X_train_p4,
    y_train_p4_c,
    lambdas_path,
    learning_rate=0.05,
    optimizer="adam",
    max_iter=5000,
)

fig, ax = plt.subplots(figsize=(7, 4.5))
for j in range(path.shape[1]):
    ax.plot(lambdas_path, path[:, j], marker="o", markersize=3, label=rf"$\theta_{{{j + 1}}}$")
ax.set_xscale("log")
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(r"$\theta_j$")
ax.set_title("Lasso coefficient path (degree 4)")
ax.legend(fontsize="small", ncol=2)
fig.tight_layout()
plt.show()

n_nonzero_small = np.sum(np.abs(path[0]) > 1e-3)
n_nonzero_large = np.sum(np.abs(path[-1]) > 1e-3)
print(
    f"nonzero coefficients: {n_nonzero_small}/4 at lambda={lambdas_path[0]:.3g}, "
    f"{n_nonzero_large}/4 at lambda={lambdas_path[-1]:.3g}"
)

## Does Lasso inherit OLS's or Ridge's convergence behavior?

Ridge's penalty adds `+2*lam*I` to the cost Hessian; the L1 penalty is
piecewise-*linear* away from the kink, so it adds **no curvature at all**.
`hessian_max_eigenvalue(X, lam=0.0)` — the *plain OLS* Hessian — is therefore the
right learning-rate reference for Lasso too, regardless of Lasso's own `lam`. The
prediction from `gradient_descent_experiment.ipynb`'s Part F: gradient descent on
Lasso should be sensitive to conditioning the way OLS was, not easy the way Ridge
was.

(Note: comparing Lasso's cost history directly against the OLS closed-form MSE, the
way Part F compared plain GD, doesn't work — `lasso_cost` always includes
`+lam*||theta||_1`, so it can never approach a pure-MSE target. The comparisons below
instead track Lasso's own cost, which is a fair, self-consistent metric.)

In [ ]:
gamma_max_ols = 2 / hessian_max_eigenvalue(X_train, lam=0.0)
lam_lasso = 0.05
optimizers = ["plain", "momentum", "adagrad", "rmsprop", "adam"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name in optimizers:
    gd = GradientDescent(
        learning_rate=0.9 * gamma_max_ols,
        lam=lam_lasso,
        max_iter=3000,
        tol=0.0,
        optimizer=name,
        penalty="l1",
    ).fit(X_train, y_train_c)
    ax.plot(np.arange(1, gd.n_iter_ + 1), gd.cost_history_, label=name)

ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("Cost")
ax.set_title(rf"Lasso convergence ($\lambda={lam_lasso:g}$, $\gamma=0.9\gamma_{{max}}^{{OLS}}$)")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

**Plain** gradient descent settles noticeably higher than the adaptive methods and
oscillates rather than smoothly converging — the subgradient at each L1 kink flips
sign abruptly as a coefficient crosses zero, and without momentum or per-coordinate
scaling, plain GD has nothing to damp that. **RMSProp** gets stuck far above
everyone else, the same pathology found for OLS/Ridge in Part F (no bias correction
on its squared-gradient average, so an early large gradient can permanently shrink
later steps). **Momentum**, **AdaGrad**, and **Adam** all reach a similar, much lower
cost band.

### Learning-rate sensitivity

Sweep the learning rate and record the cost after a fixed budget, mirroring Part E's
stability sweep — this sidesteps the cross-objective target problem above entirely,
since "final cost achieved" needs no external reference point.

In [ ]:
ratios = np.logspace(-2, 1, 13)
max_iter_sweep = 3000

fig, ax = plt.subplots(figsize=(7.5, 4.5))
with np.errstate(over="ignore", invalid="ignore"):
    for name in optimizers:
        final_costs = []
        for ratio in ratios:
            gd = GradientDescent(
                learning_rate=ratio * gamma_max_ols,
                lam=lam_lasso,
                max_iter=max_iter_sweep,
                tol=0.0,
                optimizer=name,
                penalty="l1",
            ).fit(X_train, y_train_c)
            final_costs.append(gd.cost_history_[-1])
        final_costs = np.nan_to_num(final_costs, nan=1e3, posinf=1e3)
        ax.plot(ratios, final_costs, marker="o", markersize=3, label=name)

ax.axvline(1.0, color="tab:red", ls=":", lw=1, label=r"plain GD's $\gamma_{max}$ (OLS)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"learning rate, as a multiple of plain GD's $\gamma_{max}$ (OLS Hessian)")
ax.set_ylabel(f"cost after {max_iter_sweep} steps")
ax.set_title(rf"Lasso optimizer sensitivity to learning rate ($\lambda={lam_lasso:g}$)")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

**Plain** and **momentum** both hold a low, flat cost until a sharp cliff (plain
around 2x-3x the OLS $\gamma_{max}$, momentum somewhat past that) and then blow up —
the same sharp stability boundary Part E found for OLS, since the smooth part of the
cost surface (away from any kink) is governed by the same Hessian. **AdaGrad** is
essentially flat across the *entire* range tested — remarkably robust here, a
contrast with Part F, where AdaGrad struggled on pure OLS; its per-coordinate
denominator seems to handle the noisy, sign-flipping subgradient near L1 kinks
better than OLS's smooth-but-ill-conditioned gradient. **RMSProp** degrades
smoothly (not a cliff) as the learning rate grows, consistent with its
getting-stuck-not-diverging failure mode. **Adam** stays good up to about the OLS
$\gamma_{max}$ and then degrades gradually.

So: Lasso-via-GD does inherit OLS's *sharp stability cliff* for the non-adaptive
methods (plain, and momentum eventually), exactly as the no-added-curvature argument
predicts — but it does **not** uniformly inherit OLS's difficulty for the adaptive
optimizers. AdaGrad in particular behaves better here than it did on pure OLS in
Part F, suggesting the L1 penalty's effect on the optimization *dynamics* (frequent
sign changes at the kinks) matters as much as the underlying Hessian curvature does.

## Comparison with scikit-learn

Careful with conventions: this package's `lam` and scikit-learn's `alpha` relate by
`lam = 2*alpha` (`sklearn_alpha_to_lam`; see its docstring for the derivation). No
bias term on either side: `Lasso` fits with `fit_intercept_column=False` (the
default) on the centered target above, matching scikit-learn's `fit_intercept=False`
on that same centered target, rather than either side fitting its own intercept.

In [ ]:
alpha = 0.05
lam = sklearn_alpha_to_lam(alpha)

ours = Lasso(learning_rate=0.05, lam=lam, max_iter=20000, optimizer="adam").fit(
    X_train_p4, y_train_p4_c
)
sk = SkLasso(alpha=alpha, fit_intercept=False, max_iter=100_000, tol=1e-12).fit(
    X_train_p4, y_train_p4_c
)

print(f"{'coef':>8s}  {'ours':>10s}  {'sklearn':>10s}")
for j, (a, b) in enumerate(zip(ours.coef_, sk.coef_, strict=True)):
    print(f"{f'theta_{j + 1}':>8s}  {a:>10.4f}  {b:>10.4f}")
print(f"\nmax |ours - sklearn| = {np.max(np.abs(ours.coef_ - sk.coef_)):.4f}")

Close agreement (within the print above, typically a few thousandths), confirming
`lam = 2*alpha` is the right conversion. The one systematic difference: sklearn's
coordinate descent lands on *exact* zeros for the coefficients it eliminates, while
our subgradient descent leaves small residuals nearby instead — visible in the table
above as small nonzero values where sklearn shows `0.0000`. This is expected:
subgradient methods don't have the shrink-then-clip-to-exactly-zero property that a
proximal (soft-thresholding) step has, since `lasso_subgradient`'s update is
continuous, not the piecewise projection sklearn's coordinate descent performs. It's
a genuine property of this approach, not a convergence failure — the values are
correct to within the optimization tolerance, just not exactly zero.

## Discussion: OLS, Ridge, and Lasso via gradient descent

- **Conditioning determines *which* optimizers work at all, and Lasso's answer
  depends on which ones.** Part F found only Adam (and momentum, narrowly) could fit
  degree-8 OLS in a practical budget, because OLS's Hessian condition number
  (~54,000) makes the slow eigen-direction converge far too slowly for a
  single-global-rate or naive per-coordinate update. Lasso shares that same Hessian
  away from the kinks (the L1 penalty adds zero curvature), and indeed inherits the
  sharp *stability* cliff for plain GD and momentum. But its sensitivity to specific
  optimizers is not identical to OLS's: AdaGrad, which struggled badly on smooth OLS,
  is the most robust optimizer for Lasso across the learning-rate sweep above —
  because Lasso's non-smoothness (frequent subgradient sign flips at the kinks)
  interacts with an optimizer's *update rule*, not just the underlying curvature.
- **Ridge remains the easy case.** Its `+2*lam*I` term directly raises the smallest
  Hessian eigenvalue, bringing the condition number down by roughly two orders of
  magnitude at degree 8 (Part E) — every optimizer except RMSProp reached a good fit
  in well under a thousand iterations there. Neither OLS nor Lasso gets that benefit;
  Lasso's L1 penalty shrinks and sparsifies, but doesn't touch the curvature that
  makes gradient descent slow.
- **RMSProp is the consistent weak point across all three.** OLS, Ridge, and Lasso
  all showed the same failure mode: no bias correction on its squared-gradient
  average, so it gets stuck well short of a good fit rather than diverging outright.
  Adam's bias correction (and momentum term) avoids this consistently.
- **Lasso adds a genuinely new consideration OLS/Ridge don't have**: the choice of
  subgradient at `theta_j = 0`. It doesn't change the fitted result in any way that
  matters in practice (this notebook's live demo shows the two choices differ only
  by exactly `lam`, only at coefficients that are exactly zero, which are already at
  the boundary of the feasible penalty), but it's a reminder that "the gradient" is
  not always a well-defined, unique object once a cost function stops being smooth —
  a fact that becomes central once you leave OLS/Ridge for regularizers like this
  one.